In [1]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-2-v1")   # 44.8k train rows, per assignment
print(ds)

d:\AV\Byte_Pair\for_bytepair\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [2]:
print(ds)                                 # splits and row counts
print(ds["train"][10])                    # sample row
lens = [len(t) for t in ds["train"]["text"]]
print(f"empty lines: {sum(l == 0 for l in lens):,} / {len(lens):,}")

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})
{'text': ' The game \'s battle system , the <unk> system , is carried over directly from <unk> Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ person . A character can only act once per @-@ turn , but characters can be granted multiple turns at the expense of other characters \' turns . Each character has a field and distance of movement limited by their Action <unk> . Up to nine characters can be assigned to a single mission . During gameplay , characters will call out if something happens to them , such as their health points ( HP ) getting low or being knocked out by 

In [3]:
import re

In [4]:
def clean_text(text: str) -> str:
    """Normalize and clean a single line of wikitext."""
    # remove <unk> placeholder tokens
    text = text.replace("<unk>", "")
    # remove wikitext heading markers like " = = Section = = "
    text = re.sub(r"^\s*(=\s?)+.*?(\s?=)+\s*$", "", text)
    # fix spaced punctuation artifacts of the raw dump (e.g. " ,", " .", " 's")
    text = re.sub(r"\s+([,.!?;:%)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)
    text = re.sub(r"\s+('s|'t|'re|'ve|'ll|'d|n't)\b", r"\1", text)
    # collapse repeated whitespace
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [5]:
clean_text

<function __main__.clean_text(text: str) -> str>

In [6]:
# 1. Clean the training split (batched map for speed)
train = ds["train"]
print(f"Original examples: {len(train):,}")

train_clean = train.map(
    lambda batch: {"text": [clean_text(t) for t in batch["text"]]},
    batched=True,
    # num_proc=1,
    desc="Cleaning",
)

# 2. Drop empty / whitespace-only lines left after cleaning
train_clean = train_clean.filter(
    lambda batch: [len(t) > 0 for t in batch["text"]],
    batched=True,
    # num_proc=1,
    desc="Dropping empty lines",
)
print(f"After cleaning & dropping empties: {len(train_clean):,}")

Original examples: 36,718
After cleaning & dropping empties: 17,521


In [7]:
# 3. Exact deduplication using a hash set (memory-friendly: store hashes, not strings)
import hashlib

seen = set()

def is_unique(batch):
    keep = []
    for t in batch["text"]:
        h = hashlib.md5(t.encode("utf-8")).digest()
        if h in seen:
            keep.append(False)
        else:
            seen.add(h)
            keep.append(True)
    return keep

# num_proc must be 1 here — the `seen` set is shared state
train_dedup = train_clean.filter(is_unique, batched=True, desc="Deduplicating")

print(f"After deduplication: {len(train_dedup):,}")
print(f"Removed {len(train_clean) - len(train_dedup):,} exact duplicates")

After deduplication: 17,370
Removed 151 exact duplicates


In [8]:
# Sanity check: inspect a few cleaned samples
for i in range(3):
    print(repr(train_dedup[i]["text"][:200]))
    print("---")

'Senjō no Valkyria 3: Chronicles (Japanese: 戦場のヴァルキュリア3, lit. Valkyria of the Battlefield 3), commonly referred to as Valkyria Chronicles III outside Japan, is a tactical role @-@ playing video game de'
---
'The game began development in 2010, carrying over a large portion of the work done on Valkyria Chronicles II. While it retained the standard features of the series, it also underwent multiple adjustme'
---
'It met with positive sales in Japan, and was praised by both Japanese and western critics. After release, it received downloadable content, along with an expanded edition in November of that year. It '
---


In [9]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

# 1. Build the tokenizer pipeline
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

# Pre-tokenizer: byte-level splits text into initial pieces and handles any
# unicode gracefully (same scheme as GPT-2)
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
tokenizer.decoder = decoders.ByteLevel()

In [10]:
# 2. Define the trainer with vocab size and special tokens
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

trainer = trainers.BpeTrainer(
    vocab_size=30_000,
    min_frequency=2,                 # ignore pairs appearing only once
    special_tokens=SPECIAL_TOKENS,   # reserved at ids 0-4
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=True,
)

In [11]:
# 3. Train from an iterator over the deduplicated dataset (no temp files needed)
def batch_iterator(batch_size=1000):
    for i in range(0, len(train_dedup), batch_size):
        yield train_dedup[i : i + batch_size]["text"]

tokenizer.train_from_iterator(
    batch_iterator(),
    trainer=trainer,
    length=len(train_dedup),   # enables the progress bar
)

print(f"Vocab size: {tokenizer.get_vocab_size():,}")

Vocab size: 30,000


In [12]:
# 4. Post-processing: add [CLS] ... [SEP] template (BERT-style)
cls_id = tokenizer.token_to_id("[CLS]")
sep_id = tokenizer.token_to_id("[SEP]")

tokenizer.post_processor = processors.TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B [SEP]",
    special_tokens=[("[CLS]", cls_id), ("[SEP]", sep_id)],
)

In [13]:
# 5. Save and sanity check
tokenizer.save("bpe_tokenizer.json")

enc = tokenizer.encode("The quick brown fox jumps over the lazy dog.")
print("Tokens:", enc.tokens)
print("IDs:   ", enc.ids)
print("Decoded:", tokenizer.decode(enc.ids))

Tokens: ['[CLS]', 'ĠThe', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġl', 'az', 'y', 'Ġdog', '.', '[SEP]']
IDs:    [2, 322, 2627, 4185, 5028, 13229, 612, 265, 310, 1121, 93, 6194, 18, 3]
Decoded:  The quick brown fox jumps over the lazy dog.


In [14]:
# --- Tokenizer Evaluation on WikiText-2 ---
ds2 = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

def prepare_split(split):
    """Clean and drop empty lines, same preprocessing as training."""
    texts = [clean_text(t) for t in split["text"]]
    return [t for t in texts if len(t) > 0]

val_texts = prepare_split(ds2["validation"])
test_texts = prepare_split(ds2["test"])
print(f"Validation lines: {len(val_texts):,} | Test lines: {len(test_texts):,}")

Validation lines: 1,841 | Test lines: 2,185


In [15]:
def evaluate_tokenizer(tok, texts, name):
    # batch encode (fast, parallel); disable [CLS]/[SEP] so metrics reflect pure BPE
    encodings = tok.encode_batch(texts, add_special_tokens=False)

    total_tokens = sum(len(e.ids) for e in encodings)
    total_chars  = sum(len(t) for t in texts)
    total_words  = sum(len(t.split()) for t in texts)
    unk_id = tok.token_to_id("[UNK]")
    unk_count = sum(e.ids.count(unk_id) for e in encodings)

    # tokenization consistency: encode -> decode must round-trip losslessly
    mismatches = sum(
        1 for t, e in zip(texts, encodings)
        if tok.decode(e.ids).strip() != t.strip()
    )

    metrics = {
        "split": name,
        "vocab_size": tok.get_vocab_size(),
        "lines": len(texts),
        "total_tokens": total_tokens,
        "avg_tokens_per_line": total_tokens / len(texts),
        "tokens_per_word (fertility)": total_tokens / total_words,
        "compression_ratio (chars/token)": total_chars / total_tokens,
        "unk_rate_%": 100 * unk_count / total_tokens,
        "round_trip_mismatches": mismatches,
        "consistency_%": 100 * (1 - mismatches / len(texts)),
    }
    return metrics

val_metrics  = evaluate_tokenizer(tokenizer, val_texts, "validation")
test_metrics = evaluate_tokenizer(tokenizer, test_texts, "test")

In [16]:
import pandas as pd

report = pd.DataFrame([val_metrics, test_metrics]).set_index("split").T
report

split,validation,test
vocab_size,30000.000000,30000.000000
lines,1841.000000,2185.000000
total_tokens,236036.000000,269187.000000
avg_tokens_per_line,128.210755,123.197712
tokens_per_word (fertility),1.263326,1.282057
compression_ratio (chars/token),4.644003,4.576703
unk_rate_%,0.000000,0.000000
round_trip_mismatches,0.000000,0.000000
consistency_%,100.000000,100.000000


In [17]:
# Qualitative check: how a few validation sentences tokenize
for t in val_texts[:3]:
    enc = tokenizer.encode(t, add_special_tokens=False)
    print(t[:80])
    print(enc.tokens[:20], "...")
    print(f"{len(t.split())} words -> {len(enc.ids)} tokens\n")

Homarus gammarus, known as the European lobster or common lobster, is a species 
['ĠHomarus', 'Ġgam', 'mar', 'us', ',', 'Ġknown', 'Ġas', 'Ġthe', 'ĠEuropean', 'Ġlobster', 'Ġor', 'Ġcommon', 'Ġlobster', ',', 'Ġis', 'Ġa', 'Ġspecies', 'Ġof', 'Ġcl', 'aw'] ...
121 words -> 155 tokens

Homarus gammarus is a large crustacean, with a body length up to 60 centimetres 
['ĠHomarus', 'Ġgam', 'mar', 'us', 'Ġis', 'Ġa', 'Ġlarge', 'Ġcrust', 'ace', 'an', ',', 'Ġwith', 'Ġa', 'Ġbody', 'Ġlength', 'Ġup', 'Ġto', 'Ġ60', 'Ġcentimetres', 'Ġ('] ...
108 words -> 140 tokens

The first pair of pereiopods is armed with a large, asymmetrical pair of claws. 
['ĠThe', 'Ġfirst', 'Ġpair', 'Ġof', 'Ġp', 'ere', 'i', 'opod', 's', 'Ġis', 'Ġarmed', 'Ġwith', 'Ġa', 'Ġlarge', ',', 'Ġasym', 'metrical', 'Ġpair', 'Ġof', 'Ġclaws'] ...
64 words -> 85 tokens



In [18]:
# --- Save in Hugging Face-compatible format ---
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,        # wrap the trained tokenizers.Tokenizer
    pad_token="[PAD]",
    unk_token="[UNK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
)

# saves tokenizer.json, tokenizer_config.json, special_tokens_map.json
hf_tokenizer.save_pretrained("wikitext_bpe_tokenizer")

('wikitext_bpe_tokenizer\\tokenizer_config.json',
 'wikitext_bpe_tokenizer\\tokenizer.json')

In [19]:
# --- Reload from disk (e.g. in a fresh session / another script) ---
from transformers import AutoTokenizer

reloaded = AutoTokenizer.from_pretrained("wikitext_bpe_tokenizer")
print(type(reloaded).__name__, "| vocab size:", reloaded.vocab_size)

TokenizersBackend | vocab size: 30000


In [20]:
# --- Demonstrate encoding / decoding with the reloaded tokenizer ---
text = "Byte pair encoding merges frequent character pairs into subword units."

# 1. Simple encode -> ids
ids = reloaded.encode(text)                       # includes [CLS]/[SEP]
print("IDs:    ", ids)
print("Tokens: ", reloaded.convert_ids_to_tokens(ids))

# 2. Decode back to text
print("Decoded:", reloaded.decode(ids, skip_special_tokens=True))

# 3. Full featured call — batching, padding, truncation, attention masks
batch = reloaded(
    ["Short sentence.", "A much longer sentence that will produce more tokens."],
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors="pt",
)
print("\ninput_ids:\n", batch["input_ids"])
print("attention_mask:\n", batch["attention_mask"])

IDs:     [2, 1436, 497, 3230, 2385, 15560, 2676, 2015, 8820, 1183, 9579, 639, 832, 91, 469, 2686, 18, 3]
Tokens:  ['[CLS]', 'ĠBy', 'te', 'Ġpair', 'Ġenc', 'oding', 'Ġmer', 'ges', 'Ġfrequent', 'Ġcharacter', 'Ġpairs', 'Ġinto', 'Ġsub', 'w', 'ord', 'Ġunits', '.', '[SEP]']
Decoded:  Byte pair encoding merges frequent character pairs into subword units.

input_ids:
 tensor([[   2, 4891, 5560,   18,    3,    0,    0,    0,    0,    0,    0,    0,
            0,    0],
        [   2,  306, 1316, 3394, 5560,  350, 1195, 4358,  655,  292,   79,  674,
           18,    3]])
attention_mask:
 tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
